In [ ]:
# Install required packages (needed on Colab; harmless if already installed locally)
!pip install -q openai pydantic python-dotenv requests feedparser gradio

In [ ]:
import os

# Auto-detect if we are running on Google Colab or locally
try:
    from google.colab import userdata
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    # On Colab: pull from the Secrets panel (the lock icon on the left sidebar)
    # Make sure these exact names are saved there: GROQ_API_KEY, OPENROUTER_API_KEY, PUSHOVER_USER, PUSHOVER_TOKEN
    print("Running on Colab — loading keys from Colab Secrets...")
    for key_name in ["GROQ_API_KEY", "OPENROUTER_API_KEY", "PUSHOVER_USER", "PUSHOVER_TOKEN"]:
        try:
            os.environ[key_name] = userdata.get(key_name)
            print(f"  ✅ {key_name} loaded from Colab Secrets")
        except Exception:
            print(f"  ❌ {key_name} not found in Colab Secrets — add it via the 🔑 panel")
else:
    # On local machine: load from the .env file in the project root
    from dotenv import load_dotenv
    load_dotenv(override=True)
    print("Running locally — loading keys from .env file...")
    for key_name in ["GROQ_API_KEY", "OPENROUTER_API_KEY", "PUSHOVER_USER", "PUSHOVER_TOKEN"]:
        status = "✅ Set" if os.environ.get(key_name) else "❌ Missing"
        print(f"  {key_name:30s} {status}")

# Week 8 Exercise: Deal Hunting Framework with Trust Verification

This notebook demonstrates building upon Ed Donner's Week 8 Autonomous Agent Framework. 
We use the **"OpenAI Wrapper Trick"** (learned in Week 1/2) to dynamically switch between **Groq**, **Gemini**, and **OpenRouter (Claude)** since we don't have an active OpenAI billing account.

It also introduces a new Virtual Agent into the company - the **TrustVerificationAgent**.
Before a deal is priced or a user is alerted, this agent reads the seller's reputation and the item's condition. If it smells like a scam or a broken item, the deal is thrown out immediately, ensuring the AI only buys safe, guaranteed products.

In [ ]:
import os
import json
import requests
from typing import List, Optional, Dict
from pydantic import BaseModel, Field
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

In [ ]:
# --------------------------------------------------------- #
# 1. DATA MODELS & SCHEMAS
# --------------------------------------------------------- #
class Deal(BaseModel):
    product_description: str
    price: float
    url: str
    seller_reputation: str
    condition: str

class Opportunity(BaseModel):
    deal: Deal
    estimate: float
    discount: float
    is_verified: bool = False
    verification_notes: str = ""

In [ ]:
# --------------------------------------------------------- #
# 2. THE AGENTS (The Virtual Company)
# --------------------------------------------------------- #
class Agent:
    def __init__(self):
        # Multi-provider fallback strategy
        self.providers = [
            {
                "name": "Groq",
                "client": OpenAI(api_key=os.getenv("GROQ_API_KEY", "missing"), base_url="https://api.groq.com/openai/v1"),
                "model": "llama-3.3-70b-versatile"
            },
            {
                "name": "Gemini",
                "client": OpenAI(api_key=os.getenv("GEMINI_API_KEY", "missing"), base_url="https://generativelanguage.googleapis.com/v1beta/openai/"),
                "model": "gemini-2.0-flash"
            },
            {
                "name": "OpenRouter (Claude)",
                "client": OpenAI(api_key=os.getenv("OPENROUTER_API_KEY", "missing"), base_url="https://openrouter.ai/api/v1"),
                "model": "anthropic/claude-3-5-sonnet"
            }
        ]
        
    def log(self, message: str):
        print(f"[{self.__class__.__name__}] {message}")
        
    def get_completion(self, messages: List[Dict], expect_json: bool = False) -> Optional[str]:
        """ Rotates through available keys until one works """
        for provider in self.providers:
            if provider["client"].api_key == "missing" or not provider["client"].api_key:
                continue
                
            try:
                kwargs = {
                    "model": provider["model"],
                    "messages": messages,
                }
                # Add JSON object enforcement if requested and not on OpenRouter which can be finicky
                if expect_json and provider["name"] != "OpenRouter (Claude)":
                    kwargs["response_format"] = {"type": "json_object"}
                    
                response = provider["client"].chat.completions.create(**kwargs)
                return response.choices[0].message.content.strip()
                
            except Exception as e:
                self.log(f"⚠ {provider['name']} failed: {e}. Trying next provider...")
                continue
                
        self.log("❌ ERROR: All LLM API providers failed or missing keys.")
        return None

In [ ]:
import feedparser
import re
import html
import os
import json

class ScannerAgent(Agent):
    SYSTEM_PROMPT = """You extract up to 5 of the best tech/electronics deals from messy deal website text.
    Respond strictly in valid JSON format with a single key 'deals' which is a list of objects.
    Each object must have exactly these keys:
    - 'product_description' (string: product name and key specs)
    - 'price' (float: the deal price, or 0.0 if not clearly stated)
    - 'url' (string: the deal link)
    - 'seller_reputation' (string: seller or store name with star rating if known, otherwise 'Unknown')
    - 'condition' (string: 'New', 'Refurbished', 'Used', or 'Open Box')
    Only include items with a clear price. Skip coupons, gift cards, or services."""

    FEED_URLS = [
        "https://slickdeals.net/newsearch.php?mode=frontpage&searcharea=deals&q=electronics&rss=1",
        "https://www.dealnews.com/c196/Electronics/?rss=1",
    ]
    SEEN_DEALS_FILE = "seen_deals.json"

    def __init__(self):
        super().__init__()
        self.seen_urls = self._load_seen_deals()

    def _load_seen_deals(self) -> set:
        if os.path.exists(self.SEEN_DEALS_FILE):
            try:
                with open(self.SEEN_DEALS_FILE, "r") as f:
                    return set(json.load(f))
            except Exception:
                return set()
        return set()

    def _save_seen_deals(self):
        with open(self.SEEN_DEALS_FILE, "w") as f:
            json.dump(list(self.seen_urls), f)

    def _clean_html(self, raw: str) -> str:
        raw = re.sub(r'<[^>]+>', ' ', raw)
        raw = html.unescape(raw)
        return re.sub(r'\s+', ' ', raw).strip()

    def _fetch_deals_text(self) -> str:
        lines = []
        for url in self.FEED_URLS:
            self.log(f"Fetching feed: {url[:60]}...")
            try:
                feed = feedparser.parse(url)
                for entry in feed.entries[:10]:
                    link = entry.get("link", "")
                    # Skip if we have already seen this URL!
                    if link in self.seen_urls:
                        continue
                    title = self._clean_html(entry.get("title", ""))
                    summary = self._clean_html(entry.get("summary", ""))
                    lines.append(f"- {title}: {summary[:200]}. Link: {link}")
            except Exception as e:
                self.log(f"Feed fetch failed for {url[:40]}: {e}")
        return "\n".join(lines)

    def scan(self) -> list:
        self.log("Fetching LIVE deals from real RSS feeds...")
        raw_text = self._fetch_deals_text()

        if not raw_text.strip():
            self.log("No NEW deals found in the feeds right now.")
            return []

        self.log(f"Scraped {len(raw_text.splitlines())} new distinct lines. Sending to LLM...")
        messages = [
            {"role": "system", "content": self.SYSTEM_PROMPT},
            {"role": "user", "content": f"Extract deals from this text:\n{raw_text[:3000]}\nRespond only with JSON."}
        ]

        response_text = self.get_completion(messages, expect_json=True)
        if not response_text: return []

        try:
            if response_text.startswith("```json"):
                response_text = response_text[7:-3]
            elif response_text.startswith("```"):
                response_text = response_text[3:-3]
            data = json.loads(response_text)
            deals = [Deal(**item) for item in data.get("deals", [])]
            
            # Mark all these URLs as seen so we never process them again
            for d in deals:
                self.seen_urls.add(d.url)
            self._save_seen_deals()
            
            self.log(f"Extracted {len(deals)} structured deals from live data.")
            return deals
        except Exception as e:
            self.log(f"Failed to parse JSON: {e}")
            return []


In [ ]:
class TrustVerificationAgent(Agent):
    """ The Safety Agent! Filters out scams, broken items, or bad sellers """
    
    def verify(self, deal: Deal) -> tuple[bool, str]:
        self.log(f"Verifying trust signals for: {deal.product_description[:30]}...")
        
        prompt = f"""
        Evaluate this deal for safety, authenticity, and return risk.
        Product: {deal.product_description}
        Condition: {deal.condition}
        Seller Reputation: {deal.seller_reputation}
        
        RULES:
        1. If condition indicates 'Broken', 'Cracked', 'For Parts', or 'As-Is', REJECT.
        2. If condition is 'Refurbished' OR 'Used' AND seller reputation is poor (< 4.0 stars or 'Unknown'), REJECT.
        3. If seller reputation is terrible (< 3.0 stars) regardless of condition, REJECT.
        4. Otherwise, APPROVE.
        
        Respond with EXACTLY ONE WORD first: either 'APPROVE' or 'REJECT', followed by a one-sentence reason.
        """
        
        response_text = self.get_completion([{"role": "user", "content": prompt}])
        if not response_text:
            return False, "Could not verify due to API failure."
            
        is_approved = response_text.strip().upper().startswith("APPROVE")
        self.log(f"Verdict: {response_text.strip()}")
        return is_approved, response_text.strip()

In [ ]:
class AppraiserAgent(Agent):
    """ Determines the market value of the item using the LLM directly """
    
    def estimate(self, details: str) -> float:
        print(f"   - Requesting market value for {details[:20]}...")
        prompt = f"""
        Estimate the Fair Market Value (in USD) for this item in new/like-new condition:
        {details}
        Respond WITH A SINGLE NUMBER only (e.g. 750). No dollar signs.
        """
        response_text = self.get_completion([{"role": "user", "content": prompt}])
        if not response_text: return 0.0
        
        try:
            # Clean up the response to get just the float
            import re
            numbers = re.findall(r"\d+\.\d+|\d+", response_text)
            return float(numbers[0]) if numbers else 0.0
        except:
            return 0.0

In [ ]:
class MessagingAgent(Agent):
    """ Outputs alerts to the user via Pushover """
    def alert(self, opp: Opportunity):
        print(f"\n📲 [SENDING PUSH NOTIFICATION...]")
        message = (
            f"DEAL ALERT! Profit Margin: ${opp.discount:.2f}\n"
            f"Item: {opp.deal.product_description}\n"
            f"Cost: ${opp.deal.price} | Real Value: ${opp.estimate}\n"
            f"Trust Check: PASS ({opp.verification_notes})\n"
            f"Buy Link: {opp.deal.url}"
        )
        print(message)
        
        pushover_user = os.getenv("PUSHOVER_USER")
        pushover_token = os.getenv("PUSHOVER_TOKEN")

        if not pushover_user or not pushover_token:
            print("⚠️ Pushover keys missing in .env. Skipping actual API call.")
            return

        payload = {
            "user": pushover_user,
            "token": pushover_token,
            "message": message,
            "sound": "cashregister",
        }
        
        try:
            response = requests.post("https://api.pushover.net/1/messages.json", data=payload)
            if response.status_code == 200:
                print("✅ Push notification sent successfully to your device!")
            else:
                print(f"❌ Failed to send push notification: {response.text}")
        except Exception as e:
            print(f"❌ Error sending push notification: {e}")

In [ ]:
import io
import sys
import gradio as gr
import threading
import time

# --------------------------------------------------------- #
# 3. THE PLANNING LOOP (Framework Orchestrator)
# --------------------------------------------------------- #
def run_deal_hunter_workflow():
    # Capture all print statements to a string buffer so we can show them in Gradio UI
    old_stdout = sys.stdout
    sys.stdout = capture_buf = io.StringIO()
    
    print("=== Booting Price Hunter AI (with Safety Failsafes) ===\n")
    
    intern = ScannerAgent()
    security = TrustVerificationAgent()
    appraiser = AppraiserAgent()
    secretary = MessagingAgent()
    
    deals = intern.scan()
    if not deals:
        print("No new deals found, or API failure occurred.")
        sys.stdout = old_stdout
        return capture_buf.getvalue()
        
    opportunities = []
    
    for count, deal in enumerate(deals):
        print(f"\n--- Processing Deal #{count+1} ---")
        print(f"Found: {deal.product_description}")
        
        is_safe, reason = security.verify(deal)
        if not is_safe:
            print(f"❌ SKIPPING: Failed trust check (Reason: {reason})")
            continue
            
        est_value = appraiser.estimate(deal.product_description)
        discount = est_value - deal.price
        
        if discount > 0:
            print(f"✅ APPROVED & PRICED: True value ${est_value}. Profit margin: ${discount:.2f}")
            opp = Opportunity(deal=deal, estimate=est_value, discount=discount, is_verified=True, verification_notes=reason)
            opportunities.append(opp)
            
    print("\n=======================================================")
    print("Workflow Complete. Dispatching Alerts...")
    if opportunities:
        opportunities.sort(key=lambda x: x.discount, reverse=True)
        best_deal = opportunities[0]
        if best_deal.discount > 20.0:  # Threshold for pushing directly to phone
            secretary.alert(best_deal)
        else:
            print(f"Best deal yields ${best_deal.discount:.2f}. Not worth buzzing the phone.")
    else:
        print("No safe, profitable deals found today.")
        
    sys.stdout = old_stdout
    return capture_buf.getvalue()


In [ ]:
# --------------------------------------------------------- #
# 4. GRADIO UI AND BACKGROUND TIMER EXECUTOR
# --------------------------------------------------------- #
logs_history = ""

def manual_run():
    global logs_history
    try:
        new_logs = run_deal_hunter_workflow()
        logs_history = "\n" + "*"*40 + "\n[MANUAL RUN]\n" + str(new_logs) + logs_history
    except Exception as e:
        import traceback
        err_out = traceback.format_exc()
        logs_history = "\n" + "*"*40 + f"\n[MANUAL RUN ERROR]\n{err_out}\n" + logs_history
    return logs_history

def background_worker():
    global logs_history
    while True:
        # Wait 5 minutes
        time.sleep(300)
        try:
            new_logs = run_deal_hunter_workflow()
            logs_history = "\n" + "*"*40 + "\n[TIMER EXECUTED]\n" + new_logs + logs_history
            print("Background scan completed.")
        except Exception as e:
            print(f"Background scan failed: {e}")

# Start the background thread (daemon = True so it stops when notebook is closed)
thread = threading.Thread(target=background_worker, daemon=True)
thread.start()

# Build the UI
with gr.Blocks() as demo:
    gr.Markdown("# 🤖 Autonomous Deal Hunter & Trust Verifier")
    gr.Markdown("This app automatically scrapes live RSS feeds every 5 minutes in the background, runs them through the Trust AI and Pricing AI, and pushes notifications directly to your phone via Pushover if a safe, profitable deal is found.")
    
    with gr.Row():
        run_btn = gr.Button("Force Manual Scan Now", variant="primary")
        
    # A textbox that auto-refreshes to show the latest background logs
    # Using a workaround for auto-refresh: an invisible button clicked periodically via Javascript
    output_logs = gr.Textbox(label="Agent Activity Logs", lines=25, interactive=False)
    
    run_btn.click(fn=manual_run, inputs=[], outputs=[output_logs])
    
    # The built-in load every=X is available in newer Gradio, we use it here safely
    def refresh_logs():
        return logs_history
        
    timer = gr.Timer(5)
    timer.tick(fn=refresh_logs, inputs=[], outputs=[output_logs])

# Launch the app with a public URL
demo.launch(share=True, debug=False)
